# Erzeuge Embeddings für text Chunks
Die Chunks müssen vorher erstellt und in der Tabelle bge_m3_vectors gespeichert werden

## Environment

In [ ]:
import os
os.environ["IND_PG_SCHEMA"] = "meipi-indexing"
os.environ["IND_DATA_DIR"] = "/home/padmin/Development/projekte/meipi-indexing/data"

## Import

In [ ]:
import sqlalchemy as sa
from sqlalchemy.orm import aliased
from meipi.indexing import DBOperations, DBBgeM3Vector, DBMeta
from meipi.indexing.model import ChunkItem
from meipi.indexing.config import EmbeddingConfig
from meipi.indexing.embedding_pipeline import DocItem, EmbeddingPipeline


## Konfiguration

In [ ]:
pool_id = 1
qsize = 100
num_workers = 4
emb_config = EmbeddingConfig(max_queue_size=qsize, num_workers=num_workers)


## Lies Chunks von DB

In [ ]:
numdocs = 100
V = aliased(DBMeta)
dbop = DBOperations(pool_id)
with dbop.Session() as session:
    res = session.execute(sa.select(V.id, V.inhalt).where(V.ftype == "doc").limit(numdocs)).fetchall()
    docs = [DocItem(**row._asdict()) for row in res]
print(len(docs))
print(docs[0])

## Run Pipeline

In [ ]:
pipeline = EmbeddingPipeline(emb_config,pool_id)
#pipeline.run_pipeline(chunklist)
pipeline.run_flag_pipeline(docs, create_chunks=True, test=True)


## Test

In [ ]:
from sqlalchemy.sql import null


with dbop.Session() as session:
    stmt = sa.select(sa.func.count()).select_from(DBBgeM3Vector).where(DBBgeM3Vector.content == null())
    res = session.execute(stmt).scalar_one()
    print(res)
